# Déneigeuses

Les déneigeuses, comme leurs noms l'indique, ont pour but de déblayer les rues eneigées.

Il faut donc identifier les trajets optimaux des déneigeuses en respectant le code de la route (parcourir un graphe orienté).\
Ce process doit également prendre en compte le coût de chaque équipement et devra proposer et comparer plusieurs scénarios au client.

### Informations disponibles
- Un graphe orienté pondéré avec: Des arêtes qui representent une rue entre deux intersections, avec un poids qui représenté sa distance.
  (Une idée qui avait été proposée est de mettre dans chaque noeud des coordonnées pour les calculs de distance approximatives)
- Une liste d'arêtes eneigées a parcourir (liste de tuples)

### Solutions Algorithmiques possibles

Le problème consiste à parcourir un graphe orienté pondéré en s’assurant que certaines arêtes précises sont obligatoirement parcourues.
Nous n’avons pas trouvé d’algorithme répondant spécifiquement à ce problème.
Ce problème est np-difficile et nécessite donc de trouver une solution qui trouve un compromis entre perfection du résultat obtenu et rapidité de calcul. Plusieurs algorithmes pourront nous aider:

### Edges Cases
Si un découpage statique des quartiers est choisi, alors un quartier peut être rempli de neige quand un autre est déneigé.

### Notes

In [131]:
# import image module
from IPython.display import Image

# get the image
Image(url="images/ExampleMap.webp", width=300, height=300)

In [132]:
import math
import numpy as np
import copy

In [133]:
INF = float("inf")

#### Graph initialization and display

In [134]:
#This function takes a list of (x,y,n) and replaces it by A -> B -> C by nth road for example
def display_path(l):
    if (len(l) == 0):
        return;
    print("We are on node " + chr(65 + l[0][0]))
    for (x,y,n) in l:
        route = ""
        if (n != 0):
            route = " using route " + str(n)
        print("Go from " + chr(65 + x) + " to " + chr(65 + y) + route)

a = [(0,1,0), (1,3,0), (3,4,0), (4,5,0), (5,6,0), (6,7,2)]
display_path(a)

We are on node A
Go from A to B
Go from B to D
Go from D to E
Go from E to F
Go from F to G
Go from G to H using route 2


In [135]:
city = [[[], [1], [1], [], [], [], [], [], []],
        [[1], [], [1], [1], [], [], [], [], []],
        [[1], [1], [], [1], [], [], [], [], []],
        [[], [1], [1], [], [1], [], [], [], []],
        [[], [], [], [1], [], [1], [], [], [1]],
        [[], [], [], [], [1], [], [1], [], [1]],
        [[], [], [], [], [], [1], [], [8, 1, 1], []],
        [[], [], [], [], [], [], [6, 1, 2], [], []],
        [[], [], [], [], [1], [1], [], [], []]]

#### Floyd Warshall implementation

In [136]:
# graph conversion

def graph2dist(graph):
    length = len(graph)
    res = [[INF for i in range(length)] for j in range(length)]

    for x in range(length):
        for y in range(length):
            if x == y:
                res[x][y] = 0
            elif len(graph[x][y]) != 0:
                res[x][y] = min(graph[x][y])
    return res

floyd_city = graph2dist(city)
print(floyd_city)

[[0, 1, 1, inf, inf, inf, inf, inf, inf], [1, 0, 1, 1, inf, inf, inf, inf, inf], [1, 1, 0, 1, inf, inf, inf, inf, inf], [inf, 1, 1, 0, 1, inf, inf, inf, inf], [inf, inf, inf, 1, 0, 1, inf, inf, 1], [inf, inf, inf, inf, 1, 0, 1, inf, 1], [inf, inf, inf, inf, inf, 1, 0, 1, inf], [inf, inf, inf, inf, inf, inf, 1, 0, inf], [inf, inf, inf, inf, 1, 1, inf, inf, 0]]


In [137]:
def floyd_warshall_with_paths(dist):
    V = len(dist)
    
    # Initialisation de la matrice des chemins
    path = [[[] for _ in range(V)] for _ in range(V)]

    # Initialisation des chemins directs (si arête existe)
    for i in range(V):
        for j in range(V):
            if dist[i][j] != INF and i != j:
                path[i][j] = [(i, j)]
    
    # Floyd-Warshall avec mise à jour des chemins
    for k in range(V):
        for i in range(V):
            for j in range(V):
                if dist[i][k] != INF and dist[k][j] != INF:
                    if dist[i][j] > dist[i][k] + dist[k][j]:
                        dist[i][j] = dist[i][k] + dist[k][j]
                        path[i][j] = path[i][k] + path[k][j]
    
    return dist, path

# modifie sur place donc deepcopy (temporaire)

floyd_results = floyd_warshall_with_paths(copy.deepcopy(floyd_city))
print(floyd_results)


([[0, 1, 1, 2, 3, 4, 5, 6, 4], [1, 0, 1, 1, 2, 3, 4, 5, 3], [1, 1, 0, 1, 2, 3, 4, 5, 3], [2, 1, 1, 0, 1, 2, 3, 4, 2], [3, 2, 2, 1, 0, 1, 2, 3, 1], [4, 3, 3, 2, 1, 0, 1, 2, 1], [5, 4, 4, 3, 2, 1, 0, 1, 2], [6, 5, 5, 4, 3, 2, 1, 0, 3], [4, 3, 3, 2, 1, 1, 2, 3, 0]], [[[], [(0, 1)], [(0, 2)], [(0, 1), (1, 3)], [(0, 1), (1, 3), (3, 4)], [(0, 1), (1, 3), (3, 4), (4, 5)], [(0, 1), (1, 3), (3, 4), (4, 5), (5, 6)], [(0, 1), (1, 3), (3, 4), (4, 5), (5, 6), (6, 7)], [(0, 1), (1, 3), (3, 4), (4, 8)]], [[(1, 0)], [], [(1, 2)], [(1, 3)], [(1, 3), (3, 4)], [(1, 3), (3, 4), (4, 5)], [(1, 3), (3, 4), (4, 5), (5, 6)], [(1, 3), (3, 4), (4, 5), (5, 6), (6, 7)], [(1, 3), (3, 4), (4, 8)]], [[(2, 0)], [(2, 1)], [], [(2, 3)], [(2, 3), (3, 4)], [(2, 3), (3, 4), (4, 5)], [(2, 3), (3, 4), (4, 5), (5, 6)], [(2, 3), (3, 4), (4, 5), (5, 6), (6, 7)], [(2, 3), (3, 4), (4, 8)]], [[(3, 1), (1, 0)], [(3, 1)], [(3, 2)], [], [(3, 4)], [(3, 4), (4, 5)], [(3, 4), (4, 5), (5, 6)], [(3, 4), (4, 5), (5, 6), (6, 7)], [(3, 4), (

In [138]:
# graph: [[[1, 2]]]
# targets: [(A, B, z)]

def get_weight_of_path(graph, path):
    # path is list of edge
    s = 0
    for edge in path:
        s += graph[edge[0]][edge[1]][edge[2]]
    return s

# converts a list of (A, B) edges to a list of (A, B, z)
def add_path_route(graph, path):
    res = []
    for (a, b) in path:
        if (graph[a][b] == []):
            print(a, b)
        #    break
        if len(graph[a][b]) > 0:
            res.append((a, b, graph[a][b].index(min(graph[a][b]))))
    return res

def get_shortest_path(graph, dist_floyd, curr_pos, edge):
    # edge traversal is included
    # curr_pos should not be included

    # prendre en compte le sens de la target

    target_node = -1
    # on check par ou on peut traverser l'edge

    # gets the weight of each path in order to compare them
    path1_weight = dist_floyd[0][curr_pos][edge[0]]
    path2_weight = dist_floyd[0][curr_pos][edge[1]]

    # checks for inability to travel from one node to the other
    if len(graph[edge[0]][edge[1]]) == 0:
        #print(curr_pos)
        #print(edge)
        #print(dist_floyd[1][curr_pos][edge[1]])
        
        #print(dist_floyd[1][curr_pos])
        #print(dist_floyd[0][curr_pos][edge[1]])
        return (path2_weight, add_path_route(graph, dist_floyd[1][curr_pos][edge[1]]) + [(edge[1], edge[0], edge[2])])
    elif len(graph[edge[1]][edge[0]]) == 0:
        return (path1_weight, add_path_route(graph, dist_floyd[1][curr_pos][edge[0]]) + [(edge[0], edge[1], edge[2])])

    # compares the path
    if path1_weight < path2_weight:
        return (path1_weight, add_path_route(graph, dist_floyd[1][curr_pos][edge[0]]) + [(edge[0], edge[1], edge[2])])
    return (path2_weight, add_path_route(graph, dist_floyd[1][curr_pos][edge[1]]) + [(edge[1], edge[0], edge[2])])
    # renvoie un couple avec poids du chemin et liste d'arrete
    # return (0, [edge])


In [139]:
# graph = matrice d'adjacence
def getPaths(graph, dist_floyd, targets, starting_pos):
    # on considere uniquement une seule snow plow pour le moment
    start = starting_pos[0]
    curr_pos = start
    res = []
    sp_traversal = []
    while len(targets) > 0:
        nearest_snow = (0, get_shortest_path(graph, dist_floyd, curr_pos, targets[0]))
        
        for i in range(1, len(targets)):
            tmp = get_shortest_path(graph, dist_floyd, curr_pos, targets[i])

            if tmp[0] < nearest_snow[1][0]:
                nearest_snow = (i, tmp)

        # print("pre: ", curr_pos)
        curr_pos = nearest_snow[1][1][-1][1]
        
        # print("post: ", curr_pos)
        # print(nearest_snow)
        sp_traversal += nearest_snow[1][1]
        targets.pop(nearest_snow[0])

    res.append(sp_traversal)
    return res

# ne prend pas le point le plus proche
print(getPaths(city, floyd_results, [(1, 2, 0), (7, 6, 2)], [5]))

[[(5, 6, 0), (6, 7, 2), (7, 6, 1), (6, 5, 0), (5, 4, 0), (4, 3, 0), (3, 2, 0), (2, 1, 0)]]


In [140]:
display_path(getPaths(city, floyd_results, [(1, 2, 0), (7, 6, 2)], [5])[0])

We are on node F
Go from F to G
Go from G to H using route 2
Go from H to G using route 1
Go from G to F
Go from F to E
Go from E to D
Go from D to C
Go from C to B


#### Un exemple avec un graphe correctement pondéré

In [14]:
# import image module
from IPython.display import Image

# get the image
Image(url="images/ExampleMap2.jpg", width=300, height=300)

In [15]:
#         A    B    C    D   E   F   G   H   I
city2 = [[[], [8], [5], [], [], [], [], [], []],
        [[8], [], [3], [8], [], [], [], [], []],
        [[5], [3], [], [1], [], [], [], [], []],
        [[], [8], [1], [], [1], [], [], [], []],
        [[], [], [], [1], [], [5], [], [], [8]],
        [[], [], [], [], [18], [], [4], [], [3]],
        [[], [], [], [], [], [4], [], [7, 2, 7], []],
        [[], [], [], [], [], [], [7, 2, 7], [], []],
        [[], [], [], [], [8], [3], [], [], []]]
formatted = graph2dist(city2)
floyd2 = floyd_warshall_with_paths(copy.deepcopy(formatted))
display_path(getPaths(city2, floyd2, [(1, 2, 0), (7, 6, 2)], [5])[0])

NameError: name 'graph2dist' is not defined

### HUGE TEST

In [16]:
import random

def est_connexe(matrice_adjacence):
    n = len(matrice_adjacence)
    visites = [False] * n

    def dfs(sommet):
        visites[sommet] = True
        for voisin in range(n):
            if matrice_adjacence[sommet][voisin] != [] and not visites[voisin]:
                dfs(voisin)
    dfs(0)
    return all(visites)

def two_dim_sum(l):
    s = 0
    for i in l:
        for y in i:
            s += y
    return s

def gen_random_city(n):
    connexe = False
    while not connexe:
        l = [[[] for i in range(n)] for j in range(n)]
        for i in range(n):
            for j in range(n):
                if (i != j):
                    # r = random.randint(1, 100), random.randint(0,1)
                    r = [random.randint(0, 100) for i in range(random.randint(1, 4))]
                    l[i][j] = r
                    l[j][i] = r
        for i in range(n):
            if (two_dim_sum(l[i]) == 0):
                coord = random.randint(0,n)
                if (coord == i):
                    coord = (i+1)%n
                    r = [random.randint(0, 100) for i in range(random.randint(1, 4))]
                    l[i][coord] = r
                    l[coord][i] = r
        connexe = est_connexe(l)
    return l

In [17]:
print(gen_random_city(20))

[[[], [94, 89, 75], [92, 42, 33], [24, 61, 57], [18], [9, 34, 46, 13], [71, 66, 54, 68], [60], [81, 40, 100, 99], [39], [42, 40], [44, 40, 22, 74], [69, 62], [71, 23], [70, 41], [6, 100, 39, 58], [68, 3, 12, 55], [16, 41, 87, 79], [28, 74], [42, 29]], [[94, 89, 75], [], [21, 90, 61], [63, 72, 6], [32, 64, 4, 15], [53, 86, 21], [19, 12, 10, 17], [66], [74, 65], [74], [48, 78, 53], [96, 52, 85], [29], [44, 11, 7, 40], [19], [45, 34, 28], [27, 61, 21, 14], [60, 86], [58, 24, 40, 98], [24, 66, 70, 32]], [[92, 42, 33], [21, 90, 61], [], [58, 85, 62, 56], [6, 86], [100, 67, 84], [53, 26, 18], [50], [66, 67, 54, 69], [66, 79, 92], [53, 58], [62, 97, 31], [85, 63, 4], [76], [14, 6, 23], [51, 20], [90], [13], [88, 85], [81, 59]], [[24, 61, 57], [63, 72, 6], [58, 85, 62, 56], [], [18, 74, 93], [96, 24], [20], [76, 81, 91], [50], [99, 97], [83, 91], [33, 68], [48, 4, 89], [51], [88, 85, 78, 99], [9], [57, 54, 57, 52], [1], [1, 91, 39], [3, 93, 82]], [[18], [32, 64, 4, 15], [6, 86], [18, 74, 93], 

In [30]:
import time

def get_first(g, n):
    res = []
    length = len(g)
    gap = (length // n) - 1
    for x in range(0, length, gap):
        for y in range(length):
            if (len(g[x][y]) > 0):
                res.append((x, y, random.randint(0, len(g[x][y]) - 1)))
            if (len(res) >= n):
                return res
    return res

def test_charut(n, nn):
    start = time.time()
    g = gen_random_city(n)
    stop = time.time()
    print(f"Generated graph in {(stop - start)} seconds")
    # print(g)
    start = time.time()
    proc_formatted = graph2dist(g)
    proc_floyd2 = floyd_warshall_with_paths(copy.deepcopy(proc_formatted))
    # print(get_first(g, nn))
    display_path(getPaths(g, proc_floyd2, get_first(g, nn), [5])[0])
    stop = time.time()
    print(f"Took {(stop - start)} seconds to collect {nn} patches of snow in a graph with {n} edges.")

test_charut(200, 1)

Generated graph in 0.07668328285217285 seconds
We are on node F
Go from F to h
Go from h to r using route 1
Go from r to 
Go from  to B
Go from B to A using route 3
Go from A to L
Go from L to g
Go from g to y
Go from y to h using route 1
Go from h to C using route 1
Go from C to A
Go from A to G
Go from G to t using route 2
Go from t to S using route 3
Go from S to z
Go from z to « using route 2
Go from « to D
Go from D to A
Go from A to 
Go from  to i
Go from i to E using route 1
Go from E to A
Go from A to L
Go from L to g
Go from g to y
Go from y to h using route 1
Go from h to F
Go from F to A using route 1
Go from A to G
Go from G to A
Go from A to G
Go from G to t using route 2
Go from t to S using route 3
Go from S to z
Go from z to [ using route 2
Go from [ to 
Go from  to I
Go from I to Y using route 1
Go from Y to H
Go from H to A using route 1
Go from A to G
Go from G to t using route 2
Go from t to S using route 3
Go from S to z
Go from z to [ using route 2
Go from [